In [1]:
pip install pandas numpy sentence-transformers faiss-cpu tqdm scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.metrics import silhouette_score

tqdm.pandas()

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("kcc_rice_qna_data.csv")
df.columns = ["state", "query", "answer"]

print("Initial shape:", df.shape)

Initial shape: (523911, 3)


In [4]:
def fix_token_spacing(text):
    text = re.sub(r'(\d+(\.\d+)?)([a-zA-Z]+)', r'\1 \3', text)
    text = re.sub(r'([a-zA-Z]+)(\d+)', r'\1 \2', text)
    return text

In [5]:
def normalize_units(text):
    unit_map = {
        r'\bml\b|\bmls\b': ' milliliter ',
        r'\bg\b|\bgm\b|\bgr\b': ' gram ',
        r'\bkg\b|\bkgs\b': ' kilogram ',
        r'\bl\b|\blt\b|\bltr\b|\bliter\b|\blitre\b': ' liter ',
        r'\bac\b|\bacre\b|\bacres\b': ' acre ',
        r'\bha\b|\bhectare\b': ' hectare ',
        r'\bton\b|\btons\b': ' ton ',
        r'\bpkt\b|\bpack\b|\bpacket\b': ' packet ',
        r'\bbag\b|\bbg\b': ' bag ',
        r'(\d+)\s*%': r'\1 percent'
    }

    for pattern, repl in unit_map.items():
        text = re.sub(pattern, repl, text)

    # ratios
    text = re.sub(r'(\d+)\s*milliliter\s*/\s*liter', r'\1 milliliter per liter', text)
    text = re.sub(r'(\d+)\s*gram\s*/\s*liter', r'\1 gram per liter', text)
    text = re.sub(r'(\d+)\s*kilogram\s*/\s*acre', r'\1 kilogram per acre', text)

    return text

In [6]:
import re

def normalize_language(text):
    
    patterns = {
        # 🌾 crops
        r'\bdhaan\b|\bdhan\b|\bpaddy\b': 'rice',
        
        # 🐛 pests / insects
        r'\bmachhar\b|\bmacchar\b|\bmosquito\b': 'insect',
        r'\bkeeda\b|\bkeeda\b|\bkeedaa\b': 'pest',
        
        # 🦠 disease
        r'\brog\b|\bbimari\b|\bdisease\b': 'disease',
        
        # 🌿 plant parts
        r'\bpattiya?\b|\bpattiyan\b|\bpatti\b': 'leaf',
        r'\bpaudha\b|\bpaudhe\b': 'plant',
        r'\bkhet\b': 'field',
        r'\bfasal\b': 'crop',
        r'\bbeej\b': 'seed',
        r'\bpani\b': 'water',
        
        # 🎨 COLORS (VERY IMPORTANT)
        r'\byellowing\b|\byellow\b|\bpeela\b|\bpeeli\b': 'yellow',
        r'\bred\b|\bredness\b|\breddish\b|\breding\b|\bredding\b|\blal\b': 'red',
        r'\bblack\b|\bkala\b': 'black',
        r'\bwhite\b|\bsafed\b': 'white',
        r'\bbrown\b|\bbhoora\b|\bbhura\b': 'brown',
        
        # 🌱 conditions
        r'\bsukha\b|\bdry\b': 'dry',
        r'\bgila\b|\bwet\b': 'wet',
        # 🌿 disease symptoms
        "bhuri dhabbedar": "brown spots",
        "daag": "spots",
        "pila": "yellow",
        "sukh raha": "drying",
        "murjha": "wilting",
        
        # 🌾 weeds
        "ghas": "weed",
        "khadpatwar": "weed",
        
        # 🐛 pests
        "keeda": "pest",
        "sundi": "caterpillar",
        
        # 🌱 general
        "patti": "leaf",
        "paudha": "plant",
         }

    for pattern, repl in patterns.items():
        text = re.sub(pattern, repl, text)

    return text

In [7]:
def clean_text(text):
    text = str(text)
    
    text = text.replace("\ufeff", "")
    
    text = text.lower()

    # remove noise
    noise = ["asked about", "information provided as per data"]
    for n in noise:
        text = text.replace(n, "")

    # fix tokens
    text = fix_token_spacing(text)

    # normalize units
    text = normalize_units(text)

    # normalize language
    text = normalize_language(text)

    # remove special characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # normalize spacing
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [8]:
df["clean_query"] = df["query"].progress_apply(clean_text)
df["clean_answer"] = df["answer"].progress_apply(clean_text)

df = df.dropna()
df = df[df["clean_query"].str.len() > 3]

print("After cleaning:", df.shape)

100%|██████████| 523911/523911 [00:52<00:00, 10013.20it/s]


After cleaning: (523911, 5)


In [9]:
df.to_csv("cleaned_data.csv", index=False)

In [10]:
from sklearn.model_selection import train_test_split

# Get unique queries
unique_queries = df["clean_query"].unique()

# Split queries (NOT rows)
train_q, test_q = train_test_split(unique_queries, test_size=0.2, random_state=42)

# Map back to dataframe
train_df = df[df["clean_query"].isin(train_q)].reset_index(drop=True)
test_df = df[df["clean_query"].isin(test_q)].reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (432600, 5)
Test shape: (91311, 5)


In [11]:
total = len(train_df)
unique = train_df["clean_query"].nunique()

print("Train Total:", total)
print("Train Unique:", unique)
print("Duplicate %:", (1 - unique/total)*100)

Train Total: 432600
Train Unique: 71951
Duplicate %: 83.36777623670828


In [12]:
LABELS = [
    "disease",
    "pest",
    "nutrient",
    "fertilizer",
    "irrigation",
    "other"
]

In [13]:
!pip install --upgrade openai

In [14]:
!pip install transformers torch

In [15]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0  # use GPU (if available)
)

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 9543.54it/s]


In [16]:
def label_query_local(query):
    result = classifier(query, LABELS)
    return result["labels"][0]

In [17]:
from tqdm import tqdm

queries = list(train_df["clean_query"].unique())
query_to_label = {}

batch_size = 64

for i in tqdm(range(0, len(queries), batch_size)):
    batch = queries[i:i+batch_size]

    # Add context
    batch_inputs = ["Agriculture query: " + q for q in batch]

    # 🔥 IMPORTANT: pass batch_size here
    results = classifier(
        batch_inputs,
        LABELS,
        batch_size=batch_size   # 🚀 THIS FIXES GPU USAGE
    )

    labels = [r["labels"][0] for r in results]

    for q, l in zip(batch, labels):
        query_to_label[q] = l

100%|██████████| 1125/1125 [18:34<00:00,  1.01it/s]


In [18]:
train_df["label_lvl1"] = train_df["clean_query"].map(query_to_label)

In [19]:
print(train_df["label_lvl1"].value_counts())

label_lvl1
pest          156018
disease       122995
nutrient      104838
other          41188
fertilizer      5084
irrigation      2477
Name: count, dtype: int64


In [20]:
train_df.to_csv("labeled_data.csv", index=False)

In [21]:
def refine_label_safe(query, label):
    q = query.lower()

    # 🔒 Only fix weak predictions
    if label in ["other", "nutrient"]:

        # 🐛 pest
        if any(w in q for w in [
            "borer", "hopper", "bph", "pest", "insect",
            "caterpillar", "leaf folder", "maggot"
        ]):
            return "pest"

        # 🦠 disease
        if any(w in q for w in [
            "disease", "blight", "fungal", "bacterial",
            "blast", "rust", "rot", "spot", "smut"
        ]):
            return "disease"

        # 🌱 fertilizer
        if any(w in q for w in [
            "fertilizer", "urea", "dap", "npk", "khad",
            "dose", "spray", "tricyclazole"
        ]):
            return "fertilizer"

        # 💧 irrigation
        if any(w in q for w in [
            "water", "irrigation", "watering"
        ]):
            return "irrigation"

        # 🌿 weed
        if any(w in q for w in [
            "weed", "grass", "ghas", "kharpatwar"
        ]):
            return "other"

        # 🌡️ weather/stress
        if any(w in q for w in [
            "cold", "heat", "temperature", "stress"
        ]):
            return "other"

    # ✅ keep original label if confident
    return label

In [22]:
train_df["label_lvl1"] = train_df.apply(
    lambda x: refine_label_safe(x["clean_query"], x["label_lvl1"]),
    axis=1
)

In [23]:
print(train_df["label_lvl1"].value_counts())

label_lvl1
pest          180070
disease       149375
nutrient       63683
other          25902
fertilizer     10778
irrigation      2792
Name: count, dtype: int64


In [24]:
train_df.sample(20)[["clean_query", "label_lvl1"]]

,clean_query,label_lvl1
242773,rice ki crop me sheath blight ki samsya hai,disease
334675,asking about leaf folder management in rice,pest
402130,asking about control of yellow stem borer in rice,pest
268288,how to control of soondi in rice crop,disease
393857,asking about dose of spiromesifen 229 sc for rice,fertilizer
15484,for the control of whitebacked plant hopperwph...,pest
134577,information regarding for the fert dose at the...,fertilizer
404244,asking about control of sheath rot in rice,disease
171513,rice me khera disease lga hei,disease
209365,how to control funjal disease in rice,disease


In [25]:
sample = train_df.sample(50)
print(sample[["clean_query", "label_lvl1"]])

                                              clean_query  label_lvl1
358233                         insect infestation in rice        pest
200445                        grain discoloration in rice       other
42740              how to control fungal diseases in rice     disease
61266   information rgearding for the control of sheat...     disease
10594   information regarding control of stem borer in...        pest
70167   information regarding for the control of leaf ...        pest
46494                   how to control blast of rice crop        pest
48874   farmer want to know information about how to c...     disease
137539  information regarding for the control of sheat...     disease
338764                          hwa se ganna gir gaya hai       other
125509  farmer want to know howto control fungal infec...     disease
259392  farmer wants to know information about how to ...       other
28534                       rice me chuho ke liye kay kre    nutrient
353222    asking abo

In [26]:
from sentence_transformers import SentenceTransformer

# load model
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


# encode
embeddings = model.encode(
    train_df["clean_query"].tolist(),
    batch_size=64,          # adjust if GPU/CPU
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # IMPORTANT for clustering/RAG
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10308.47it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 6760/6760 [01:29<00:00, 75.68it/s] 


In [27]:
# queries = train_df["clean_query"].tolist()

# embeddings = model.encode(
#     queries,
#     batch_size=128,
#     show_progress_bar=True
# )
# # 

In [28]:
import numpy as np

np.save("embeddings.npy", embeddings)

In [29]:
print(embeddings.shape)

(432600, 384)


In [30]:
import pandas as pd

train_df = pd.read_csv("labeled_data.csv")

In [31]:
import numpy as np

embeddings = np.load("embeddings.npy")

In [32]:
import os
os.environ["OMP_NUM_THREADS"] = "2"

In [33]:
!pip install hdbscan

In [34]:
import pandas as pd
import numpy as np
import hdbscan
from sklearn.preprocessing import normalize

In [35]:

train_df = pd.read_csv("labeled_data.csv")
embeddings = np.load("embeddings.npy")

In [36]:
embeddings = normalize(embeddings)

In [37]:
sample_size = 20000  # safe size

idx = np.random.choice(len(train_df), sample_size, replace=False)

emb_sample = embeddings[idx]
df_sample = train_df.iloc[idx].copy()

In [113]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=50,
    metric='euclidean'
    
)

sample_labels = clusterer.fit_predict(emb_sample)

df_sample["cluster"] = sample_labels

In [39]:
print("Unique clusters:", np.unique(sample_labels))

Unique clusters: [-1  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22
 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46
 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70
 71 72 73]


In [40]:
cluster_centers = {}

for c in np.unique(sample_labels):
    if c == -1:
        continue
    
    cluster_centers[c] = emb_sample[sample_labels == c].mean(axis=0)

In [41]:
from sklearn.metrics.pairwise import cosine_similarity

train_df["hdbscan_cluster"] = -1

centers = np.array(list(cluster_centers.values()))
cluster_ids = list(cluster_centers.keys())

for i in range(0, len(embeddings), 5000):  # batch processing
    
    batch_emb = embeddings[i:i+5000]
    
    sim = cosine_similarity(batch_emb, centers)
    
    closest = sim.argmax(axis=1)
    
    train_df.iloc[i:i+5000, train_df.columns.get_loc("hdbscan_cluster")] = [
        cluster_ids[j] for j in closest
    ]

In [42]:
for c in train_df["hdbscan_cluster"].unique()[:10]:
    print("\nCluster:", c)
    print(
        train_df[train_df["hdbscan_cluster"] == c]["clean_query"]
        .head(5)
        .tolist()
    )


Cluster: 48
['weed management in direct seeded rice', 'weed management in rice', 'rice weed management', 'white tip in rice weeding', 'asking about the weeds problem in rice field']

Cluster: 1
['chloripyriphos dosage', 'wanted toknow the amount of furadan need for 1 bigha', 'asking about fertilzer dose', 'asking about fertilzer dose', 'aasking about the copntroll measure of balst']

Cluster: 8
['leaf folder', 'information regarding how to control of f folder in paddylea', 'how to control leaf caterpiler in pabby', 'control of algel in apdy nursery', 'leaf eating caterpiler control']

Cluster: 42
['control of weeeds in rice and control of rats', 'leaf miner sucking pest', 'sucking pest and stem borer', 'pest attack in rice field', 'asking about the pest attack in rice field']

Cluster: 21
['control of gallmidge in rice', 'nutrient management in rice', 'asking about the control of puk in rice', 'asking about the rice nutrient management', 'asking about the control measure of rice gandh

In [117]:
from sklearn.metrics import silhouette_score

idx = np.random.choice(len(train_df), 3000, replace=False)

score = silhouette_score(
    embeddings[idx],
    train_df.iloc[idx]["hdbscan_cluster"]
)

print("Silhouette:", score)

IndexError: index 392081 is out of bounds for axis 0 with size 71951

In [44]:
def cluster_purity(df):
    total = len(df)
    correct = 0

    for cluster in df["hdbscan_cluster"].unique():
        subset = df[df["hdbscan_cluster"] == cluster]
        
        if len(subset) == 0:
            continue
        
        majority = subset["label_lvl1"].value_counts().max()
        correct += majority

    return correct / total

print("Cluster Purity:", cluster_purity(train_df))

Cluster Purity: 0.8302288488210818


In [45]:
sample = train_df.sample(50)

correct = 0

for _, row in sample.iterrows():
    print(row["clean_query"], "→", row["hdbscan_cluster"])
    
    # manually mark correct / incorrect

# calculate manually

dose of propiconazole 25 ec in rice → 20
green leafhopper control in rice → 44
grain discolouration in rice → 25
how to control fungal disease in rice → 26
control of gundhi bug in rice → 42
how to control funjal disease in rice → 11
brown plant hopper in rice → 46
information regarding how to control fungal disease in rice → 26
information regarding for the control of termite attack in paddyjhona crop → 12
control of sheath blight in rice → 60
information regarding stem borer in rice → 70
weed control in rice nursery → 48
asking about control of neck blast in rice → 17
bacterial leaf blight in rice → 35
plant protection in lime tree → 7
bacterial leaf blight in rice → 35
rice me khar patwar hai → 50
rice me bal such rhi hai → 50
rice stem borer spray quinalphos 2 mllitre → 63
information rgearding for the control of sheath blight attack on paddyjhona crop → 5
green leafhopper on rice → 44
rice ke jado me kida lag raha hai → 50
farmer want to how know to control blast of rice crop → 33

In [46]:
final_df = train_df[[
    "clean_query",
    "clean_answer",
    "label_lvl1",
    "hdbscan_cluster"   # or hdbscan_cluster
]].copy()

# remove duplicates
final_df = final_df.drop_duplicates(subset=["clean_query"]).reset_index(drop=True)

In [47]:
print(len(final_df), embeddings.shape)

71951 (432600, 384)


In [48]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    final_df["clean_query"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10240.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1125/1125 [00:10<00:00, 103.64it/s]


In [49]:
print(len(final_df), embeddings.shape)

71951 (71951, 384)


In [50]:
import faiss
import numpy as np

dim = embeddings.shape[1]

# cosine similarity (best for text)
index = faiss.IndexFlatIP(dim)

index.add(embeddings)

print("Total vectors in index:", index.ntotal)

Total vectors in index: 71951


In [51]:
faiss.write_index(index, "faiss_index.idx")
final_df.to_pickle("rag_data.pkl")

In [52]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10615.36it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [53]:
def retrieve(query, top_k=5):
    
    q_emb = model.encode([query], normalize_embeddings=True)
    
    D, I = index.search(q_emb, top_k)
    
    results = final_df.iloc[I[0]]
    
    return results[["clean_query", "clean_answer"]]

In [54]:
print(retrieve("rice me stem borer ka ilaj"))

                               clean_query  \
51013    rice me stem borer ki janakari de   
4448   rice me stem borer ke liye kya kare   
29962           rice me stem borer ke liye   
53829  rice ke field me stem borer ki dawa   
69594  rice me stem borer ke liye kya dale   

                                            clean_answer  
51013  kisan bhai aap rice me cartap hydrochloride 50...  
4448                             trichocard lga sakte ho  
29962                   cartap ya fipronil ka praog kare  
53829  catop hydrochloride 50 sp 500 gram prati acre ...  
69594                    chlorantraniliprole 04 gram 4 6  


In [55]:
print(retrieve("why more water is required"))

                                        clean_query  \
11920                         excess water in field   
35479                        water holding capacity   
26393  increase aeration and water holding capacity   
25795                        eradicate excess water   
8994                  asking about water management   

                                            clean_answer  
11920  recommended eradicate water from field opt for...  
35479  improve clay particals to improve water holdin...  
26393  recommended apply compost 2 tonacre to increas...  
25795              use pumpset to eradicate excess water  
8994   irrigation water is to be applied to maintain ...  


In [82]:
!pip install networkx transformers sentence-transformers accelerate

In [92]:
import networkx as nx
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [93]:
G = nx.DiGraph()

In [94]:
cluster_centroids = {}

for cluster in final_df["hdbscan_cluster"].unique():
    
    idx = final_df[final_df["hdbscan_cluster"] == cluster].index.tolist()
    
    emb = embeddings[idx]
    
    cluster_centroids[cluster] = np.mean(emb, axis=0)

In [95]:
cluster_keywords = {}

for cluster in final_df["hdbscan_cluster"].unique():
    
    cluster_data = final_df[final_df["hdbscan_cluster"] == cluster]
    
    texts = cluster_data["clean_query"].tolist()
    
    if len(texts) < 5:
        continue
    
    vectorizer = TfidfVectorizer(
        max_features=5,
        ngram_range=(1,2),   # better than single words
        stop_words="english"
    )
    
    X = vectorizer.fit_transform(texts)
    
    keywords = vectorizer.get_feature_names_out()
    
    cluster_keywords[cluster] = list(keywords)

In [96]:
for i, row in final_df.iterrows():
    
    query = row["clean_query"]
    answer = row["clean_answer"]
    cluster = row["hdbscan_cluster"]
    
    cluster_node = f"cluster_{cluster}"
    
    # -------- NODES --------
    
    G.add_node(cluster_node, type="cluster")
    G.add_node(query, type="query")
    G.add_node(answer, type="answer")
    
    # -------- EDGES --------
    
    G.add_edge(query, answer, relation="has_solution")
    G.add_edge(query, cluster_node, relation="belongs_to")
    G.add_edge(cluster_node, answer, relation="cluster_solution")
    
    # -------- KEYWORD ENTITIES --------
    
    if cluster in cluster_keywords:
        for kw in cluster_keywords[cluster]:
            
            G.add_node(kw, type="entity")
            
            G.add_edge(kw, cluster_node, relation="represents")
            G.add_edge(kw, query, relation="appears_in")

In [101]:
from sklearn.metrics.pairwise import cosine_similarity

def kg_lookup(query):
    
    q_emb = model.encode([query], normalize_embeddings=True)
    
    best_cluster = None
    best_sim = -1
    
    for cluster, centroid in cluster_centroids.items():
        
        sim = cosine_similarity(q_emb, centroid.reshape(1, -1))[0][0]
        
        if sim > best_sim:
            best_sim = sim
            best_cluster = cluster
    
    cluster_node = f"cluster_{best_cluster}"
    
    answers = []
    
    for neighbor in G.neighbors(cluster_node):
        if G.nodes[neighbor]["type"] == "answer":
            answers.append(neighbor)
    
    return answers[:5]

In [102]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12238.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [103]:
result = kg_lookup("rice me stem borer ka ilaj")
print(result)

['advised him to apply classic 20 2 milliliter liter of water', 'suggested to spray hitcel 1 mllitre of water', 'suggested to spray rocket 2 mllitre of water', 'suggested to spray carbendezim 1 gmlitre or hexaconazole 2 mllitre of waternext spray after 10 days', 'advised to apply chloropyriphos 20 ec 2 milliliter per liter of water or monocrotophos 40 ec 3 milliliter per liter of water']


In [105]:
import pickle

with open("kg_final.pkl", "wb") as f:
    pickle.dump(G, f)

In [106]:
cluster_centroids = {}

for cluster in final_df["hdbscan_cluster"].unique():
    
    idx = final_df[final_df["hdbscan_cluster"] == cluster].index.tolist()
    
    emb = embeddings[idx]
    
    centroid = np.mean(emb, axis=0)
    
    cluster_centroids[cluster] = centroid

In [107]:
print(len(cluster_centroids))

74


In [108]:
import pickle

with open("centroids.pkl", "wb") as f:
    pickle.dump(cluster_centroids, f)